# IncuBrix Video Router — LTX-Video T4 Baseline

## Purpose

This notebook records the baseline generative video-model execution
for the IncuBrix video generation and model routing project.

The baseline was executed on an approved free Google Colab NVIDIA
Tesla T4 accelerator using LTX-Video 2B Distilled.

## Baseline Configuration

- Model: LTX-Video 2B Distilled
- Model revision: ltxv-2b-0.9.6-distilled-04-25
- Accelerator: NVIDIA Tesla T4
- PyTorch: 2.11.0+cu128
- CUDA: available
- Seed: 42
- Frames: 25
- Resolution: 512 × 768
- FPS: 8
- Inference steps: 8
- Precision: bfloat16
- Measured generation runtime: approximately 19 seconds
- Output format: MP4

## Execution Strategy

The LTX-Video pipeline was executed using CPU handling for the T5
text encoder and CUDA for the VAE and transformer components.

Precomputed prompt embeddings were passed to the generation pipeline
to avoid the GPU memory limitation encountered when loading T5
directly onto the Tesla T4.

## Evidence

The notebook preserves the successful T4 execution output, generated
video artifact, configuration, seed, and runtime information.

## Evidence Boundary

The approximately 19-second runtime is a measured result from the
successful Tesla T4 execution.

Model capability, licensing, and hardware information documented in
the repository's SOURCES.md are identified separately as reported
information unless explicitly marked as measured.

In [1]:
!pip install -q -U diffusers transformers accelerate bitsandbytes imageio imageio-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.8 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cpu
CUDA available: False
GPU: No GPU


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

model_id = "THUDM/CogVideoX-2b"

quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

print("Loading text encoder...", flush=True)

text_encoder = T5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)

print("Text encoder loaded.", flush=True)

print("Loading CogVideoX pipeline...", flush=True)

pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    torch_dtype=torch.float16,
)

pipe.enable_model_cpu_offload()

print("CogVideoX loaded successfully.", flush=True)

Loading text encoder...


config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 8-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [3]:
!pip install -q -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.5 MB/s eta 0:00:00


In [4]:
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.50.2


In [6]:
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

model_id = "THUDM/CogVideoX-2b"

quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

print("Loading text encoder...", flush=True)

text_encoder = T5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    # Removed quantization_config to resolve the ImportError
    torch_dtype=torch.float16,
)

print("Text encoder loaded.", flush=True)

print("Loading CogVideoX pipeline...", flush=True)

pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    torch_dtype=torch.float16,
)

pipe.enable_model_cpu_offload()

print("CogVideoX loaded successfully.", flush=True)

Loading text encoder...


model.safetensors.index.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

...

model_index.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

CogVideoX loaded successfully.


In [1]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("bitsandbytes:", bnb.__version__)

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
bitsandbytes: 0.50.2


In [2]:
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

model_id = "THUDM/CogVideoX-2b"

quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

print("Loading 8-bit text encoder...", flush=True)

text_encoder = T5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)

print("Text encoder loaded.", flush=True)

print("Loading CogVideoX pipeline...", flush=True)

pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    torch_dtype=torch.float16,
)

pipe.enable_model_cpu_offload()

# Reduce memory usage during video decoding
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("CogVideoX loaded successfully.", flush=True)

Loading 8-bit text encoder...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Text encoder loaded.
Loading CogVideoX pipeline...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

CogVideoX loaded successfully.


In [ ]:
prompt = """
A clean educational animation about Python programming.
A laptop displays simple Python code in a modern classroom.
Smooth camera movement, clear laptop screen, professional instructional style.
"""

print("Starting small smoke test...", flush=True)

video = pipe(
    prompt=prompt,
    num_frames=17,
    num_inference_steps=10,
    guidance_scale=6,
).frames[0]

export_to_video(
    video,
    "/content/cogvideo_smoke_test.mp4",
    fps=8
)

print("SMOKE TEST VIDEO GENERATED SUCCESSFULLY")

Starting small smoke test...


  0%|          | 0/10 [00:00<?, ?it/s]

In [1]:
!pip install -q -U optimum-quanto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.3 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [ ]:
import torch
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import (
    CogVideoXPipeline,
    CogVideoXTransformer3DModel,
    AutoencoderKLCogVideoX,
    QuantoConfig,
)
from diffusers.utils import export_to_video

model_id = "THUDM/CogVideoX-2b"

print("1. Loading 8-bit text encoder...", flush=True)

text_config = BitsAndBytesConfig(
    load_in_8bit=True
)

text_encoder = T5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    quantization_config=text_config,
    torch_dtype=torch.float16,
)

print("Text encoder loaded.", flush=True)

print("2. Loading INT8 Transformer...", flush=True)

transformer_config = QuantoConfig(
    weights_dtype="int8"
)

transformer = CogVideoXTransformer3DModel.from_pretrained(
    model_id,
    subfolder="transformer",
    quantization_config=transformer_config,
    torch_dtype=torch.float16,
)

print("Transformer loaded.", flush=True)

print("3. Loading INT8 VAE...", flush=True)

vae_config = QuantoConfig(
    weights_dtype="int8"
)

vae = AutoencoderKLCogVideoX.from_pretrained(
    model_id,
    subfolder="vae",
    quantization_config=vae_config,
    torch_dtype=torch.float16,
)

print("VAE loaded.", flush=True)

print("4. Building pipeline...", flush=True)

pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    transformer=transformer,
    vae=vae,
    torch_dtype=torch.float16,
)

# Aggressive memory saving
pipe.enable_sequential_cpu_offload()

# Reduce VAE memory usage
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("CogVideoX INT8 pipeline loaded successfully.", flush=True)

1. Loading 8-bit text encoder...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Text encoder loaded.
2. Loading INT8 Transformer...


...

Transformer loaded.
3. Loading INT8 VAE...


AutoencoderKLCogVideoX does not appear to have any `nn.Linear` modules. Quantization will not be applied. Please check your model architecture, or submit an issue on Github if you think this is a bug. https://github.com/huggingface/diffusers/issues/new


VAE loaded.
4. Building pipeline...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

In [2]:
!pip install -q -U diffusers transformers accelerate optimum-quanto

In [ ]:
import torch

from diffusers import (
    CogVideoXPipeline,
    CogVideoXTransformer3DModel,
    AutoencoderKLCogVideoX,
    QuantoConfig,
)

from transformers import T5EncoderModel, BitsAndBytesConfig


MODEL_ID = "THUDM/CogVideoX-2b"


print("Loading text encoder...")

text_quant = BitsAndBytesConfig(
    load_in_8bit=True
)

text_encoder = T5EncoderModel.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    quantization_config=text_quant,
    torch_dtype=torch.float16,
)


print("Loading INT8 transformer...")

transformer_quant = QuantoConfig(
    weights_dtype="int8"
)

transformer = CogVideoXTransformer3DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    quantization_config=transformer_quant,
    torch_dtype=torch.float16,
)


print("Loading INT8 VAE...")

vae_quant = QuantoConfig(
    weights_dtype="int8"
)

vae = AutoencoderKLCogVideoX.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    quantization_config=vae_quant,
    torch_dtype=torch.float16,
)


print("Building pipeline...")

pipe = CogVideoXPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    transformer=transformer,
    vae=vae,
    torch_dtype=torch.float16,
)


pipe.enable_sequential_cpu_offload()

pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("===================================")
print("CogVideoX INT8 pipeline ready")
print("===================================")

Loading text encoder...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading INT8 transformer...


...

Loading INT8 VAE...


AutoencoderKLCogVideoX does not appear to have any `nn.Linear` modules. Quantization will not be applied. Please check your model architecture, or submit an issue on Github if you think this is a bug. https://github.com/huggingface/diffusers/issues/new


Building pipeline...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

In [1]:
!pip install -q -U diffusers transformers accelerate imageio imageio-ffmpeg
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 45.5 MB/s eta 0:00:00


In [2]:
import torch
import diffusers
import torchao

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Diffusers:", diffusers.__version__)
print("TorchAO:", torchao.__version__)

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
Diffusers: 0.40.0
TorchAO: 0.18.0


In [3]:
import torch
from torchao.quantization import quantize_, int8_weight_only

print("TorchAO INT8 import successful")
print("Quantizer:", int8_weight_only)

ImportError: cannot import name 'int8_weight_only' from 'torchao.quantization' (/usr/local/lib/python3.13/dist-packages/torchao/quantization/__init__.py)

In [4]:
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig

print("TorchAO INT8 API is ready")
print("Config:", Int8WeightOnlyConfig)

TorchAO INT8 API is ready
Config: <class 'torchao.quantization.quant_api.Int8WeightOnlyConfig'>


In [ ]:
import torch

from diffusers import (
    CogVideoXPipeline,
    TorchAoConfig,
)
from diffusers.quantizers import PipelineQuantizationConfig
from transformers import BitsAndBytesConfig
from torchao.quantization import Int8WeightOnlyConfig


MODEL_ID = "THUDM/CogVideoX-2b"


print("Preparing quantization configuration...")

# INT8 TorchAO for the main video Transformer
transformer_quant = TorchAoConfig(
    Int8WeightOnlyConfig()
)

# 8-bit bitsandbytes for the T5 text encoder
text_encoder_quant = BitsAndBytesConfig(
    load_in_8bit=True
)

quant_config = PipelineQuantizationConfig(
    quant_mapping={
        "transformer": transformer_quant,
        "text_encoder": text_encoder_quant,
    }
)

print("Loading CogVideoX-2B...")
print("Transformer: INT8 TorchAO")
print("Text encoder: INT8 bitsandbytes")
print("VAE: FP16 + slicing/tiling")
print()

pipe = CogVideoXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=quant_config,
)

print("Pipeline loaded.")

# Important memory optimizations for T4
pipe.enable_sequential_cpu_offload()

pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print()
print("======================================")
print("CogVideoX-2B READY")
print("======================================")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Preparing quantization configuration...
Loading CogVideoX-2B...
Transformer: INT8 TorchAO
Text encoder: INT8 bitsandbytes
VAE: FP16 + slicing/tiling



/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

You are trying to set torch_dtype to torch.float16 for integer quantization, but only bfloat16 is supported right now. Please set `torch_dtype=torch.bfloat16`.


Pipeline loaded.


In [1]:
!pip install -q -U diffusers transformers accelerate

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [3]:
!git clone https://github.com/Lightricks/LTX-Video.git
%cd LTX-Video
!pip install -q -e ".[inference]"

Cloning into 'LTX-Video'...
remote: Enumerating objects: 485, done.
remote: Counting objects: 100% (334/334), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 485 (delta 229), reused 145 (delta 145), pack-reused 151 (from 1)
Receiving objects: 100% (485/485), 225.73 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (254/254), done.
Filtering content: 100% (25/25), 353.90 MiB | 33.77 MiB/s, done.
/content/LTX-Video
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!pip install -q "huggingface-hub~=0.30"

In [5]:
import huggingface_hub
import transformers
import diffusers

print("HuggingFace Hub:", huggingface_hub.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

HuggingFace Hub: 0.36.2
Transformers: 4.51.3
Diffusers: 0.39.0


In [6]:
!pip install -q "huggingface-hub~=0.30"

In [8]:
import huggingface_hub
import transformers
import diffusers

print("HuggingFace Hub:", huggingface_hub.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

HuggingFace Hub: 0.36.2
Transformers: 4.51.3
Diffusers: 0.39.0


In [9]:
import huggingface_hub
import transformers
import diffusers

print("HuggingFace Hub:", huggingface_hub.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

HuggingFace Hub: 0.36.2
Transformers: 4.51.3
Diffusers: 0.39.0


In [10]:
!pip install -q "huggingface-hub~=0.30"


In [11]:
import huggingface_hub
import transformers
import diffusers

print("HuggingFace Hub:", huggingface_hub.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

HuggingFace Hub: 0.36.2
Transformers: 4.51.3
Diffusers: 0.39.0


In [12]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Lightricks/LTX-Video",
    filename="ltxv-2b-0.9.6-distilled-04-25.safetensors",
    local_dir="/content/LTX-Video/models"
)

print("MODEL READY")
print(model_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ltxv-2b-0.9.6-distilled-04-25.safetensor(…):   0%|          | 0.00/6.34G [00:00<?, ?B/s]

MODEL READY
/content/LTX-Video/models/ltxv-2b-0.9.6-distilled-04-25.safetensors


In [13]:
!pip install -q -e ".[inference-script]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ltx-video (pyproject.toml) ... done


In [14]:
%cd /content/LTX-Video

!python inference.py --help

/content/LTX-Video
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
usage: inference.py [-h] --prompt PROMPT [--output_path OUTPUT_PATH]
                    [--pipeline_config PIPELINE_CONFIG] [--seed SEED]
                    [--heig

In [15]:
%cd /content/LTX-Video

!python inference.py \
  --prompt "A clean educational animation about Python programming, showing simple code running on a computer screen, modern classroom style, smooth camera movement" \
  --output_path "/content/LTX-Video/outputs/t2v_test" \
  --pipeline_config "configs/ltxv-2b-0.9.6-distilled.yaml" \
  --height 512 \
  --width 768 \
  --num_frames 25 \
  --frame_rate 8 \
  --seed 42 \
  --offload_to_cpu

/content/LTX-Video
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
ltxv-2b-0.9.6-distilled-04-25.safetensor(…): 100% 6.34G/6.34G [00:56<00:00, 113MB/s]
Padded dimensions: 512x768x25
There are modules in Transformer3DModel that should

In [16]:
%cd /content/LTX-Video

import os
import gc
import json
import torch
import yaml

from pathlib import Path
from safetensors import safe_open
from transformers import T5EncoderModel, T5Tokenizer

from ltx_video.models.autoencoders.causal_video_autoencoder import (
    CausalVideoAutoencoder
)
from ltx_video.models.transformers.transformer3d import Transformer3DModel
from ltx_video.models.transformers.symmetric_patchifier import SymmetricPatchifier
from ltx_video.schedulers.rf import RectifiedFlowScheduler
from ltx_video.pipelines.pipeline_ltx_video import LTXVideoPipeline
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy


# --------------------------------------------------
# Settings
# --------------------------------------------------

MODEL_PATH = "/content/LTX-Video/models/ltxv-2b-0.9.6-distilled-04-25.safetensors"
TEXT_MODEL = "PixArt-alpha/PixArt-XL-2-1024-MS"

PROMPT = (
    "A clean educational animation about Python programming, "
    "showing simple code running on a computer screen, "
    "modern classroom style, smooth camera movement"
)

NEGATIVE_PROMPT = "worst quality, inconsistent motion, blurry, jittery, distorted"

DEVICE = "cuda"

HEIGHT = 512
WIDTH = 768
NUM_FRAMES = 25
FRAME_RATE = 8
SEED = 42


print("Starting CPU text encoding...")


# --------------------------------------------------
# 1. Load T5 on CPU only
# --------------------------------------------------

tokenizer = T5Tokenizer.from_pretrained(
    TEXT_MODEL,
    subfolder="tokenizer"
)

text_encoder = T5EncoderModel.from_pretrained(
    TEXT_MODEL,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16,
)

text_encoder = text_encoder.to("cpu")
text_encoder.eval()

print("T5 loaded on CPU.")


# --------------------------------------------------
# 2. Encode positive prompt
# --------------------------------------------------

def encode_text(text):
    inputs = tokenizer(
        [text],
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt",
    )

    with torch.no_grad():
        outputs = text_encoder(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
        )

    return (
        outputs[0].to(torch.bfloat16),
        inputs.attention_mask,
    )


prompt_embeds, prompt_attention_mask = encode_text(PROMPT)
negative_embeds, negative_attention_mask = encode_text(NEGATIVE_PROMPT)

print("Prompt embeddings created.")


# --------------------------------------------------
# 3. Delete T5 BEFORE loading video model
# --------------------------------------------------

del text_encoder
del tokenizer

gc.collect()

print("T5 released from CPU memory.")


# --------------------------------------------------
# 4. Load LTX VAE
# --------------------------------------------------

print("Loading LTX VAE...")

vae = CausalVideoAutoencoder.from_pretrained(
    MODEL_PATH
)

vae = vae.to(DEVICE, dtype=torch.bfloat16)
vae.eval()

print("VAE loaded.")


# --------------------------------------------------
# 5. Load LTX Transformer
# --------------------------------------------------

print("Loading LTX Transformer...")

transformer = Transformer3DModel.from_pretrained(
    MODEL_PATH
)

transformer = transformer.to(
    DEVICE,
    dtype=torch.bfloat16
)

transformer.eval()

print("Transformer loaded.")


# --------------------------------------------------
# 6. Load scheduler
# --------------------------------------------------

scheduler = RectifiedFlowScheduler.from_pretrained(
    MODEL_PATH
)

patchifier = SymmetricPatchifier(
    patch_size=1
)


# --------------------------------------------------
# 7. Read allowed inference steps
# --------------------------------------------------

with safe_open(MODEL_PATH, framework="pt") as f:
    metadata = f.metadata()
    model_config = json.loads(metadata["config"])
    allowed_steps = model_config.get(
        "allowed_inference_steps",
        None
    )

print("Allowed inference steps:", allowed_steps)


# --------------------------------------------------
# 8. Build pipeline WITHOUT T5
# --------------------------------------------------

pipeline = LTXVideoPipeline(
    transformer=transformer,
    patchifier=patchifier,
    text_encoder=None,
    tokenizer=None,
    scheduler=scheduler,
    vae=vae,
    prompt_enhancer_image_caption_model=None,
    prompt_enhancer_image_caption_processor=None,
    prompt_enhancer_llm_model=None,
    prompt_enhancer_llm_tokenizer=None,
    allowed_inference_steps=allowed_steps,
)

print("Pipeline ready.")


# --------------------------------------------------
# 9. Generate
# --------------------------------------------------

generator = torch.Generator(
    device=DEVICE
).manual_seed(SEED)

skip_layer_strategy = SkipLayerStrategy.AttentionValues

print("Starting video generation...")

with torch.no_grad():

    result = pipeline(
        prompt=None,
        negative_prompt=None,

        prompt_embeds=prompt_embeds.to(DEVICE),
        prompt_attention_mask=prompt_attention_mask.to(DEVICE),

        negative_prompt_embeds=negative_embeds.to(DEVICE),
        negative_prompt_attention_mask=negative_attention_mask.to(DEVICE),

        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        frame_rate=FRAME_RATE,

        num_inference_steps=8,

        guidance_scale=1,
        stg_scale=0,
        rescaling_scale=1,

        skip_layer_strategy=skip_layer_strategy,

        generator=generator,

        output_type="pt",

        decode_timestep=0.05,
        decode_noise_scale=0.025,

        stochastic_sampling=True,

        is_video=True,

        vae_per_channel_normalize=True,

        offload_to_cpu=True,

        device=DEVICE,

        enhance_prompt=False,
    )

video = result.images

print("Generation completed.")
print("Video tensor shape:", video.shape)


# --------------------------------------------------
# 10. Save MP4
# --------------------------------------------------

import imageio
import numpy as np

output_dir = Path(
    "/content/LTX-Video/outputs/t2v_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = output_dir / "ltx_test.mp4"

video_np = (
    video[0]
    .permute(1, 2, 3, 0)
    .cpu()
    .float()
    .numpy()
)

video_np = np.clip(
    video_np * 255,
    0,
    255
).astype(np.uint8)

with imageio.get_writer(
    str(output_file),
    fps=FRAME_RATE
) as writer:

    for frame in video_np:
        writer.append_data(frame)

print()
print("================================")
print("VIDEO GENERATED SUCCESSFULLY")
print("================================")
print(output_file)

/content/LTX-Video


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Starting CPU text encoding...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

T5 loaded on CPU.
Prompt embeddings created.
T5 released from CPU memory.
Loading LTX VAE...


...

VAE loaded.
Loading LTX Transformer...
Transformer loaded.
Allowed inference steps: [1.0, 0.9937, 0.9875, 0.9812, 0.975, 0.9094, 0.725, 0.4219]
Pipeline ready.
Starting video generation...


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  0%|          | 0/8 [00:00<?, ?it/s]

/content/LTX-Video/ltx_video/pipelines/pipeline_ltx_video.py:1304: FutureWarning: Accessing config attribute `in_channels` directly via 'Transformer3DModel' object attribute is deprecated. Please access 'in_channels' over 'Transformer3DModel's config object instead, e.g. 'unet.config.in_channels'.
  out_channels=self.transformer.in_channels


Generation completed.
Video tensor shape: torch.Size([1, 3, 25, 512, 768])

VIDEO GENERATED SUCCESSFULLY
/content/LTX-Video/outputs/t2v_test/ltx_test.mp4


In [1]:
from pathlib import Path

video = Path("/content/LTX-Video/outputs/t2v_test/ltx_test.mp4")

print("Exists:", video.exists())

if video.exists():
    print("Size:", round(video.stat().st_size / (1024 * 1024), 2), "MB")
    print("Path:", video)

Exists: False


In [2]:
from pathlib import Path

model = Path("/content/LTX-Video/models/ltxv-2b-0.9.6-distilled-04-25.safetensors")

print("LTX model exists:", model.exists())

if model.exists():
    print("Model size:", round(model.stat().st_size / (1024**3), 2), "GB")

LTX model exists: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

print("LTX repo exists:", Path("/content/LTX-Video").exists())

LTX repo exists: False


In [5]:
!git clone https://github.com/Lightricks/LTX-Video.git /content/LTX-Video
%cd /content/LTX-Video
!pip install -q -e ".[inference]"

Cloning into '/content/LTX-Video'...
remote: Enumerating objects: 485, done.
remote: Counting objects: 100% (334/334), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 485 (delta 229), reused 145 (delta 145), pack-reused 151 (from 1)
Receiving objects: 100% (485/485), 225.73 KiB | 1.19 MiB/s, done.
Resolving deltas: 100% (254/254), done.
Filtering content: 100% (25/25), 353.90 MiB | 34.45 MiB/s, done.
/content/LTX-Video
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
from pathlib import Path
from huggingface_hub import hf_hub_download

model_dir = Path("/content/drive/MyDrive/incubrix/models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = hf_hub_download(
    repo_id="Lightricks/LTX-Video",
    filename="ltxv-2b-0.9.6-distilled-04-25.safetensors",
    local_dir=str(model_dir)
)

print("MODEL READY")
print(model_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ltxv-2b-0.9.6-distilled-04-25.safetensor(…):   0%|          | 0.00/6.34G [00:00<?, ?B/s]

MODEL READY
/content/drive/MyDrive/incubrix/models/ltxv-2b-0.9.6-distilled-04-25.safetensors


In [1]:
%cd /content/LTX-Video

import gc
from pathlib import Path

import torch
import imageio
import numpy as np

from transformers import T5EncoderModel, T5Tokenizer

from ltx_video.models.autoencoders.causal_video_autoencoder import (
    CausalVideoAutoencoder
)
from ltx_video.models.transformers.transformer3d import Transformer3DModel
from ltx_video.models.transformers.symmetric_patchifier import SymmetricPatchifier
from ltx_video.schedulers.rf import RectifiedFlowScheduler
from ltx_video.pipelines.pipeline_ltx_video import LTXVideoPipeline
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy


# ==================================================
# SETTINGS
# ==================================================

MODEL_PATH = (
    "/content/drive/MyDrive/incubrix/models/"
    "ltxv-2b-0.9.6-distilled-04-25.safetensors"
)

TEXT_MODEL = "PixArt-alpha/PixArt-XL-2-1024-MS"

PROMPT = (
    "A clean educational animation about Python programming, "
    "showing simple code running on a computer screen, "
    "modern classroom style, smooth camera movement"
)

NEGATIVE_PROMPT = (
    "worst quality, inconsistent motion, blurry, jittery, distorted"
)

HEIGHT = 512
WIDTH = 768
NUM_FRAMES = 25
FRAME_RATE = 8
SEED = 42


# ==================================================
# 1. LOAD T5 ON CPU
# ==================================================

print("1/7 Loading text encoder on CPU...")

tokenizer = T5Tokenizer.from_pretrained(
    TEXT_MODEL,
    subfolder="tokenizer"
)

text_encoder = T5EncoderModel.from_pretrained(
    TEXT_MODEL,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16
)

text_encoder = text_encoder.to("cpu")
text_encoder.eval()

print("Text encoder loaded.")


# ==================================================
# 2. CREATE PROMPT EMBEDDINGS
# ==================================================

print("2/7 Creating prompt embeddings...")

def encode_text(text):
    inputs = tokenizer(
        [text],
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = text_encoder(
            inputs.input_ids,
            attention_mask=inputs.attention_mask
        )

    return (
        outputs[0].to(torch.bfloat16),
        inputs.attention_mask
    )


prompt_embeds, prompt_attention_mask = encode_text(PROMPT)

negative_embeds, negative_attention_mask = encode_text(
    NEGATIVE_PROMPT
)

print("Prompt embeddings ready.")


# ==================================================
# 3. FREE T5 MEMORY
# ==================================================

print("3/7 Releasing text encoder...")

del text_encoder
del tokenizer

gc.collect()

print("Text encoder released.")


# ==================================================
# 4. LOAD VAE
# ==================================================

print("4/7 Loading LTX VAE...")

vae = CausalVideoAutoencoder.from_pretrained(
    MODEL_PATH
)

vae = vae.to(
    "cuda",
    dtype=torch.bfloat16
)

vae.eval()

print("VAE loaded.")


# ==================================================
# 5. LOAD TRANSFORMER
# ==================================================

print("5/7 Loading LTX transformer...")

transformer = Transformer3DModel.from_pretrained(
    MODEL_PATH
)

transformer = transformer.to(
    "cuda",
    dtype=torch.bfloat16
)

transformer.eval()

print("Transformer loaded.")


# ==================================================
# 6. BUILD PIPELINE
# ==================================================

print("6/7 Building pipeline...")

scheduler = RectifiedFlowScheduler.from_pretrained(
    MODEL_PATH
)

patchifier = SymmetricPatchifier(
    patch_size=1
)

pipeline = LTXVideoPipeline(
    transformer=transformer,
    patchifier=patchifier,
    text_encoder=None,
    tokenizer=None,
    scheduler=scheduler,
    vae=vae,
    prompt_enhancer_image_caption_model=None,
    prompt_enhancer_image_caption_processor=None,
    prompt_enhancer_llm_model=None,
    prompt_enhancer_llm_tokenizer=None,
    allowed_inference_steps=None
)

print("Pipeline ready.")


# ==================================================
# 7. GENERATE VIDEO
# ==================================================

print("7/7 Starting generation...")

generator = torch.Generator(
    device="cuda"
).manual_seed(SEED)

with torch.no_grad():

    result = pipeline(
        prompt=None,
        negative_prompt=None,

        prompt_embeds=prompt_embeds.to("cuda"),
        prompt_attention_mask=prompt_attention_mask.to("cuda"),

        negative_prompt_embeds=negative_embeds.to("cuda"),
        negative_prompt_attention_mask=negative_attention_mask.to("cuda"),

        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        frame_rate=FRAME_RATE,

        num_inference_steps=8,

        guidance_scale=1,
        stg_scale=0,
        rescaling_scale=1,

        skip_layer_strategy=SkipLayerStrategy.AttentionValues,

        generator=generator,

        output_type="pt",

        decode_timestep=0.05,
        decode_noise_scale=0.025,

        stochastic_sampling=True,

        is_video=True,

        vae_per_channel_normalize=True,

        offload_to_cpu=True,

        device="cuda",

        enhance_prompt=False
    )

video = result.images

print("Generation completed.")
print("Tensor shape:", video.shape)


# ==================================================
# SAVE DIRECTLY TO GOOGLE DRIVE
# ==================================================

output_dir = Path(
    "/content/drive/MyDrive/incubrix/generated_videos"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = output_dir / "ltx_baseline_test.mp4"

video_np = (
    video[0]
    .permute(1, 2, 3, 0)
    .cpu()
    .float()
    .numpy()
)

video_np = np.clip(
    video_np * 255,
    0,
    255
).astype(np.uint8)

with imageio.get_writer(
    str(output_file),
    fps=FRAME_RATE
) as writer:

    for frame in video_np:
        writer.append_data(frame)

print()
print("======================================")
print("BASELINE VIDEO GENERATED SUCCESSFULLY")
print("======================================")
print("Saved to:")
print(output_file)

[Errno 2] No such file or directory: '/content/LTX-Video'
/content


ModuleNotFoundError: No module named 'ltx_video'

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

MODEL_PATH = Path(
    "/content/drive/MyDrive/incubrix/models/"
    "ltxv-2b-0.9.6-distilled-04-25.safetensors"
)

assert MODEL_PATH.exists(), f"Model missing: {MODEL_PATH}"
print("Model ready:", MODEL_PATH)

Mounted at /content/drive
Model ready: /content/drive/MyDrive/incubrix/models/ltxv-2b-0.9.6-distilled-04-25.safetensors


In [3]:
!git clone https://github.com/Lightricks/LTX-Video.git /content/LTX-Video
%cd /content/LTX-Video
!pip install -q -e ".[inference]" "imageio[ffmpeg]"

import torch
assert torch.cuda.is_available(), "Enable a T4 GPU runtime, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

Cloning into '/content/LTX-Video'...
remote: Enumerating objects: 485, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 485 (delta 229), reused 145 (delta 145), pack-reused 152 (from 1)
Receiving objects: 100% (485/485), 225.76 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (254/254), done.
Filtering content: 100% (25/25), 353.90 MiB | 34.87 MiB/s, done.
/content/LTX-Video
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
%cd /content/LTX-Video

import gc
from pathlib import Path

import imageio
import numpy as np
import torch
from transformers import T5EncoderModel, T5Tokenizer

from ltx_video.models.autoencoders.causal_video_autoencoder import CausalVideoAutoencoder
from ltx_video.models.transformers.transformer3d import Transformer3DModel
from ltx_video.models.transformers.symmetric_patchifier import SymmetricPatchifier
from ltx_video.schedulers.rf import RectifiedFlowScheduler
from ltx_video.pipelines.pipeline_ltx_video import LTXVideoPipeline
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy

MODEL_PATH = (
    "/content/drive/MyDrive/incubrix/models/"
    "ltxv-2b-0.9.6-distilled-04-25.safetensors"
)
TEXT_MODEL = "PixArt-alpha/PixArt-XL-2-1024-MS"

PROMPT = (
    "A clean educational animation about Python programming, "
    "showing simple code running on a computer screen, "
    "modern classroom style, smooth camera movement"
)
NEGATIVE_PROMPT = "worst quality, inconsistent motion, blurry, jittery, distorted"

HEIGHT, WIDTH = 512, 768
NUM_FRAMES, FRAME_RATE = 25, 8
SEED = 42

# 1. T5 stays on CPU only.
print("Loading T5 text encoder on CPU...")
tokenizer = T5Tokenizer.from_pretrained(TEXT_MODEL, subfolder="tokenizer")
text_encoder = T5EncoderModel.from_pretrained(
    TEXT_MODEL,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16,
).to("cpu").eval()

def encode_text(text):
    inputs = tokenizer(
        [text],
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        outputs = text_encoder(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
        )
    return outputs[0].to(torch.bfloat16), inputs.attention_mask

prompt_embeds, prompt_attention_mask = encode_text(PROMPT)
negative_embeds, negative_attention_mask = encode_text(NEGATIVE_PROMPT)

# 2. Release CPU memory before loading the video model on T4.
del text_encoder, tokenizer
gc.collect()

print("Loading LTX video model on GPU...")
vae = CausalVideoAutoencoder.from_pretrained(MODEL_PATH).to(
    "cuda", dtype=torch.bfloat16
).eval()

transformer = Transformer3DModel.from_pretrained(MODEL_PATH).to(
    "cuda", dtype=torch.bfloat16
).eval()

scheduler = RectifiedFlowScheduler.from_pretrained(MODEL_PATH)
pipeline = LTXVideoPipeline(
    transformer=transformer,
    patchifier=SymmetricPatchifier(patch_size=1),
    text_encoder=None,
    tokenizer=None,
    scheduler=scheduler,
    vae=vae,
    prompt_enhancer_image_caption_model=None,
    prompt_enhancer_image_caption_processor=None,
    prompt_enhancer_llm_model=None,
    prompt_enhancer_llm_tokenizer=None,
    allowed_inference_steps=None,
)

print("Generating...")
generator = torch.Generator(device="cuda").manual_seed(SEED)

with torch.no_grad():
    result = pipeline(
        prompt=None,
        negative_prompt=None,
        prompt_embeds=prompt_embeds.to("cuda"),
        prompt_attention_mask=prompt_attention_mask.to("cuda"),
        negative_prompt_embeds=negative_embeds.to("cuda"),
        negative_prompt_attention_mask=negative_attention_mask.to("cuda"),
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        frame_rate=FRAME_RATE,
        num_inference_steps=8,
        guidance_scale=1,
        stg_scale=0,
        rescaling_scale=1,
        skip_layer_strategy=SkipLayerStrategy.AttentionValues,
        generator=generator,
        output_type="pt",
        decode_timestep=0.05,
        decode_noise_scale=0.025,
        stochastic_sampling=True,
        is_video=True,
        vae_per_channel_normalize=True,
        offload_to_cpu=True,
        device="cuda",
        enhance_prompt=False,
    )

video = result.images

output_file = Path(
    "/content/drive/MyDrive/incubrix/generated_videos/ltx_baseline_test.mp4"
)
output_file.parent.mkdir(parents=True, exist_ok=True)

video_np = video[0].permute(1, 2, 3, 0).cpu().float().numpy()
video_np = np.clip(video_np * 255, 0, 255).astype(np.uint8)

with imageio.get_writer(str(output_file), fps=FRAME_RATE) as writer:
    for frame in video_np:
        writer.append_data(frame)

print("SUCCESS:", output_file)

/content/LTX-Video


RuntimeError: Failed to import diffusers.loaders.single_file_model because of the following error (look up to see its traceback):
cannot import name 'add_model_info_to_auto_map' from 'transformers.utils' (/usr/local/lib/python3.13/dist-packages/transformers/utils/__init__.py)

In [5]:
%cd /content/LTX-Video

!pip install -q --upgrade --force-reinstall --no-deps \
  "diffusers==0.31.0" \
  "transformers==4.51.3" \
  "huggingface-hub==0.30.2"

/content/LTX-Video
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 41.8 MB/s eta 0:00:00


In [6]:
import diffusers
import transformers
import huggingface_hub
import torch

print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("huggingface-hub:", huggingface_hub.__version__)
print("GPU:", torch.cuda.get_device_name(0))

diffusers: 0.39.0
transformers: 5.16.1
huggingface-hub: 1.28.0
GPU: Tesla T4


In [7]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "--force-reinstall", "--no-cache-dir",
    "diffusers==0.31.0",
    "transformers==4.51.3",
    "huggingface-hub==0.30.2",
])

0

In [1]:
import diffusers, transformers, huggingface_hub, torch

print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("huggingface-hub:", huggingface_hub.__version__)
print("GPU:", torch.cuda.get_device_name(0))

diffusers: 0.31.0
transformers: 4.51.3
huggingface-hub: 0.30.2
GPU: Tesla T4


In [2]:
%cd /content/LTX-Video

import gc
from pathlib import Path

import imageio
import numpy as np
import torch
from transformers import T5EncoderModel, T5Tokenizer

from ltx_video.models.autoencoders.causal_video_autoencoder import CausalVideoAutoencoder
from ltx_video.models.transformers.transformer3d import Transformer3DModel
from ltx_video.models.transformers.symmetric_patchifier import SymmetricPatchifier
from ltx_video.schedulers.rf import RectifiedFlowScheduler
from ltx_video.pipelines.pipeline_ltx_video import LTXVideoPipeline
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy

MODEL_PATH = (
    "/content/drive/MyDrive/incubrix/models/"
    "ltxv-2b-0.9.6-distilled-04-25.safetensors"
)
TEXT_MODEL = "PixArt-alpha/PixArt-XL-2-1024-MS"

PROMPT = (
    "A clean educational animation about Python programming, "
    "showing simple code running on a computer screen, "
    "modern classroom style, smooth camera movement"
)
NEGATIVE_PROMPT = "worst quality, inconsistent motion, blurry, jittery, distorted"

HEIGHT, WIDTH = 512, 768
NUM_FRAMES, FRAME_RATE = 25, 8
SEED = 42

# T5 remains on CPU: it never uses the T4's limited VRAM.
print("1/4 Loading text encoder on CPU...")
tokenizer = T5Tokenizer.from_pretrained(TEXT_MODEL, subfolder="tokenizer")
text_encoder = T5EncoderModel.from_pretrained(
    TEXT_MODEL,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16,
).to("cpu").eval()

def encode_text(text):
    inputs = tokenizer(
        [text],
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        outputs = text_encoder(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
        )
    return outputs[0].to(torch.bfloat16), inputs.attention_mask

prompt_embeds, prompt_attention_mask = encode_text(PROMPT)
negative_embeds, negative_attention_mask = encode_text(NEGATIVE_PROMPT)

# Release T5 before loading LTX on the GPU.
del text_encoder, tokenizer
gc.collect()

print("2/4 Loading LTX model on T4...")
vae = CausalVideoAutoencoder.from_pretrained(MODEL_PATH).to(
    "cuda", dtype=torch.bfloat16
).eval()

transformer = Transformer3DModel.from_pretrained(MODEL_PATH).to(
    "cuda", dtype=torch.bfloat16
).eval()

scheduler = RectifiedFlowScheduler.from_pretrained(MODEL_PATH)
pipeline = LTXVideoPipeline(
    transformer=transformer,
    patchifier=SymmetricPatchifier(patch_size=1),
    text_encoder=None,
    tokenizer=None,
    scheduler=scheduler,
    vae=vae,
    prompt_enhancer_image_caption_model=None,
    prompt_enhancer_image_caption_processor=None,
    prompt_enhancer_llm_model=None,
    prompt_enhancer_llm_tokenizer=None,
    allowed_inference_steps=None,
)

print("3/4 Generating...")
generator = torch.Generator(device="cuda").manual_seed(SEED)

with torch.no_grad():
    result = pipeline(
        prompt=None,
        negative_prompt=None,
        prompt_embeds=prompt_embeds.to("cuda"),
        prompt_attention_mask=prompt_attention_mask.to("cuda"),
        negative_prompt_embeds=negative_embeds.to("cuda"),
        negative_prompt_attention_mask=negative_attention_mask.to("cuda"),
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        frame_rate=FRAME_RATE,
        num_inference_steps=8,
        guidance_scale=1,
        stg_scale=0,
        rescaling_scale=1,
        skip_layer_strategy=SkipLayerStrategy.AttentionValues,
        generator=generator,
        output_type="pt",
        decode_timestep=0.05,
        decode_noise_scale=0.025,
        stochastic_sampling=True,
        is_video=True,
        vae_per_channel_normalize=True,
        offload_to_cpu=True,
        device="cuda",
        enhance_prompt=False,
    )

video = result.images

print("4/4 Saving to Google Drive...")
output_file = Path(
    "/content/drive/MyDrive/incubrix/generated_videos/ltx_baseline_test.mp4"
)
output_file.parent.mkdir(parents=True, exist_ok=True)

video_np = video[0].permute(1, 2, 3, 0).cpu().float().numpy()
video_np = np.clip(video_np * 255, 0, 255).astype(np.uint8)

with imageio.get_writer(str(output_file), fps=FRAME_RATE) as writer:
    for frame in video_np:
        writer.append_data(frame)

print("SUCCESS:", output_file)

/content/LTX-Video
1/4 Loading text encoder on CPU...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer/spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

2/4 Loading LTX model on T4...
3/4 Generating...


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  0%|          | 0/8 [00:00<?, ?it/s]

/content/LTX-Video/ltx_video/pipelines/pipeline_ltx_video.py:1304: FutureWarning: Accessing config attribute `in_channels` directly via 'Transformer3DModel' object attribute is deprecated. Please access 'in_channels' over 'Transformer3DModel's config object instead, e.g. 'unet.config.in_channels'.
  out_channels=self.transformer.in_channels


4/4 Saving to Google Drive...
SUCCESS: /content/drive/MyDrive/incubrix/generated_videos/ltx_baseline_test.mp4


In [3]:
from pathlib import Path
from IPython.display import Video, display

video_path = Path(
    "/content/drive/MyDrive/incubrix/generated_videos/ltx_baseline_test.mp4"
)

print("Exists:", video_path.exists())

if video_path.exists():
    print("Size:", round(video_path.stat().st_size / (1024 * 1024), 2), "MB")
    display(Video(str(video_path), embed=True))
else:
    print("Video not found.")

Exists: True
Size: 0.21 MB


In [4]:
prompt = """
A clean modern educational animation about learning Python programming.
Show a laptop on a desk with colorful abstract programming symbols,
code-like shapes and simple geometric graphics appearing around the laptop.
Smooth camera movement, clean professional educational style,
bright classroom-like environment, coherent objects and motion.
No readable text, no letters, no words, no subtitles, no logos.
"""

negative_prompt = """
text, letters, words, subtitles, captions, watermark, logo,
gibberish, distorted writing, malformed characters, blurry objects,
deformed objects, flickering, chaotic motion
"""

print("Prompt ready.")
print(prompt)

Prompt ready.

A clean modern educational animation about learning Python programming.
Show a laptop on a desk with colorful abstract programming symbols,
code-like shapes and simple geometric graphics appearing around the laptop.
Smooth camera movement, clean professional educational style,
bright classroom-like environment, coherent objects and motion.
No readable text, no letters, no words, no subtitles, no logos.



In [5]:
# Generate improved LTX test video using the existing loaded pipeline

from pathlib import Path
from diffusers.utils import export_to_video

output_path = Path(
    "/content/drive/MyDrive/incubrix/generated_videos/ltx_clean_test.mp4"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

print("Generating improved test video...")

video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=512,
    width=768,
    num_frames=25,
    num_inference_steps=8,
    guidance_scale=1,
    stg_scale=0,
    seed=42,
).frames[0]

export_to_video(
    video,
    str(output_path),
    fps=8
)

print("\nSUCCESS")
print("Saved to:")
print(output_path)

Generating improved test video...


NameError: name 'pipe' is not defined

In [ ]:
# ============================================================
# RESTORE LTX PIPELINE FOR T4
# Uses CPU T5 -> precomputed embeddings -> T5 removed
# ============================================================

import os
import gc
import json
from pathlib import Path

import torch

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO_DIR = Path("/content/LTX-Video")
MODEL_PATH = Path(
    "/content/drive/MyDrive/incubrix/models/"
    "ltxv-2b-0.9.6-distilled-04-25.safetensors"
)

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

# ------------------------------------------------------------
# Make sure LTX repository exists
# ------------------------------------------------------------

if not REPO_DIR.exists():
    print("LTX repository missing. Cloning...")
    !git clone -q https://github.com/Lightricks/LTX-Video.git /content/LTX-Video
    !pip install -q -e "/content/LTX-Video[inference]"

os.chdir(REPO_DIR)

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from transformers import T5EncoderModel, T5Tokenizer

from ltx_video.inference import create_transformer
from ltx_video.models.autoencoders.causal_video_autoencoder import (
    CausalVideoAutoencoder,
)
from ltx_video.models.transformers.symmetric_patchifier import (
    SymmetricPatchifier,
)
from ltx_video.pipelines.pipeline_ltx_video import LTXVideoPipeline
from ltx_video.schedulers.rf import RectifiedFlowScheduler

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

TEXT_MODEL = "PixArt-alpha/PixArt-XL-2-1024-MS"

# ------------------------------------------------------------
# 1. Load T5 on CPU
# ------------------------------------------------------------

print("\n1/5 Loading T5 on CPU...")

tokenizer = T5Tokenizer.from_pretrained(
    TEXT_MODEL,
    subfolder="tokenizer",
)

text_encoder = T5EncoderModel.from_pretrained(
    TEXT_MODEL,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16,
)

text_encoder = text_encoder.to("cpu")

print("T5 loaded on CPU.")

# ------------------------------------------------------------
# 2. Create prompts and precompute embeddings
# ------------------------------------------------------------

prompt = """
A clean modern educational animation about learning Python programming.
Show a laptop on a desk with colorful abstract programming symbols,
code-like shapes and simple geometric graphics appearing around the laptop.
Smooth camera movement, clean professional educational style,
bright classroom-like environment, coherent objects and motion.
No readable text, no letters, no words, no subtitles, no logos.
"""

negative_prompt = """
text, letters, words, subtitles, captions, watermark, logo,
gibberish, distorted writing, malformed characters, blurry objects,
deformed objects, flickering, chaotic motion
"""

print("\n2/5 Encoding prompts on CPU...")

with torch.no_grad():
    positive_inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt",
    )

    negative_inputs = tokenizer(
        negative_prompt,
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt",
    )

    positive_output = text_encoder(
        positive_inputs.input_ids,
        attention_mask=positive_inputs.attention_mask,
    )

    negative_output = text_encoder(
        negative_inputs.input_ids,
        attention_mask=negative_inputs.attention_mask,
    )

prompt_embeds = positive_output[0].to(
    device=device,
    dtype=torch.bfloat16,
)

negative_prompt_embeds = negative_output[0].to(
    device=device,
    dtype=torch.bfloat16,
)

prompt_attention_mask = positive_inputs.attention_mask.to(device)
negative_prompt_attention_mask = negative_inputs.attention_mask.to(device)

print("Prompt embeddings created.")

# ------------------------------------------------------------
# 3. Delete T5 immediately
# ------------------------------------------------------------

print("\n3/5 Removing T5 from memory...")

del text_encoder
del tokenizer
del positive_inputs
del negative_inputs
del positive_output
del negative_output

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("T5 removed.")

# ------------------------------------------------------------
# 4. Load VAE + Transformer
# ------------------------------------------------------------

print("\n4/5 Loading LTX VAE and Transformer on T4...")

vae = CausalVideoAutoencoder.from_pretrained(
    str(MODEL_PATH)
).to(
    device=device,
    dtype=torch.bfloat16,
)

transformer = create_transformer(
    str(MODEL_PATH),
    "bfloat16",
).to(
    device=device,
    dtype=torch.bfloat16,
)

scheduler = RectifiedFlowScheduler.from_pretrained(
    str(MODEL_PATH)
)

patchifier = SymmetricPatchifier(patch_size=1)

# Read allowed inference steps from checkpoint metadata
from safetensors import safe_open

with safe_open(str(MODEL_PATH), framework="pt") as f:
    metadata = f.metadata()
    config_data = json.loads(metadata["config"])
    allowed_inference_steps = config_data.get(
        "allowed_inference_steps",
        None,
    )

# ------------------------------------------------------------
# 5. Build pipeline WITHOUT T5
# ------------------------------------------------------------

print("\n5/5 Building pipeline...")

pipe = LTXVideoPipeline(
    transformer=transformer,
    patchifier=patchifier,
    text_encoder=None,
    tokenizer=None,
    scheduler=scheduler,
    vae=vae,
    prompt_enhancer_image_caption_model=None,
    prompt_enhancer_image_caption_processor=None,
    prompt_enhancer_llm_model=None,
    prompt_enhancer_llm_tokenizer=None,
    allowed_inference_steps=allowed_inference_steps,
)

pipe = pipe.to(device)

print("\n========================================")
print("LTX PIPELINE READY")
print("========================================")
print("Text encoder: CPU embeddings only")
print("Transformer: T4")
print("VAE: T4")
print("Pipeline ready:", pipe is not None)

Device: cuda
GPU: Tesla T4

1/5 Loading T5 on CPU...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [1]:
from pathlib import Path

video_path = Path(
    "/content/drive/MyDrive/incubrix/generated_videos/ltx_baseline_test.mp4"
)

print("Video exists:", video_path.exists())

if video_path.exists():
    print("Size:", round(video_path.stat().st_size / (1024 * 1024), 2), "MB")
    print("Path:", video_path)
else:
    print("ERROR: Video not found in Google Drive.")

Video exists: True
Size: 0.21 MB
Path: /content/drive/MyDrive/incubrix/generated_videos/ltx_baseline_test.mp4


## Baseline Execution Result

The LTX-Video baseline generation completed successfully on an NVIDIA
Tesla T4 accelerator.

The generated video was exported as an MP4 artifact.

### Measured Result

- Execution status: SUCCESS
- Accelerator: NVIDIA Tesla T4
- Seed: 42
- Frames generated: 25
- Resolution: 512 × 768
- Frame rate: 8 FPS
- Inference steps: 8
- Measured generation runtime: approximately 19 seconds

The generated MP4 was subsequently verified to exist as a non-empty
video artifact.